# MIVA-KNIGHT: Complete Algorithm Implementation
## Multimodal Interactive Voice Assistant — Taxonomy, Pipelines A–E

**Author:** Oluwakayode (Kayode) Soyinka | IT 581 Capstone | Concordia University of Edmonton  
**Supervisor:** Dr. Baidya Saha

---

This notebook implements **every algorithm** from the thesis *MIVA-KNIGHT: Complete Thesis — Multimodal Interactive Voice Assistant with Knowledge-driven Hybrid Intelligence and Adaptive Domain Transfer*.

### Algorithms Covered
| # | Algorithm | Section |
|---|---|---|
| 1 | Joint Multimodal Representation | §3.1 |
| 2 | Coordinated Representation (Symmetric InfoNCE) | §3.2 |
| 3 | Missing-Modality Robust Inference | §3.3 |
| 4 | Model-Based Fusion (Cross-Modal Transformer) | §3.4 |
| 5 | Early Fusion | §3.5 |
| 6 | Late Fusion | §3.6 |
| 7 | TextEncoderV2 (BERT + TextProjection) | §4 |
| 8 | Pipeline A Training Epoch (InfoNCE) | §4 |
| 9 | PMI Knowledge Graph Construction | §5 |
| 10 | KG Depth-First Neighbour Search | §5 |
| 11 | ROCO Fine-tune (Surgical Domain Transfer) | §6 |
| 12 | NER KG Re-ranker | §7 |
| 13 | Full Query Pipeline | §7 |
| 14 | RAVDESS Embedding Cache | §8 |
| 15 | SupCon Loss | §8 |
| 16 | SupCon Training Epoch | §8 |
| 17 | Centroid Nearest-Neighbour Evaluation | §8 |
| 18 | CREMA-D Middle-Frame Extraction | §9 |
| 19 | Pipeline D Stage 1 — InfoNCE Training | §9 |
| 20 | Fusion Transformer Forward Pass | §10 |
| 21 | Pipeline E Training Loop | §10 |
| 22 | Domain Hot-Swap | §11 |

---
### Notebook Structure
Every **code cell** is preceded by a **markdown cell** that explains:
- The algorithm in **technical language** (mathematical equations, theorems, lemmas)
- The algorithm in **plain language** (simple analogy)
- A **flowchart** (ASCII/Matplotlib) where helpful

## 0. Environment Setup & Imports

### What this cell does (plain language)
This is the toolbox. Before we can build anything, we need to import the right libraries — PyTorch for neural networks, NumPy for maths, Matplotlib for plots, and HuggingFace Transformers for pre-trained models like BERT and Wav2Vec.

### Shared mathematical foundation
All MIVA-KNIGHT algorithms operate in the **shared embedding space**:
$$\mathbb{S}^{511} = \{v \in \mathbb{R}^{512} \mid \|v\|_2 = 1\}$$

**Axiom (Metric Consistency):** For any $\hat{u}, \hat{v} \in \mathbb{S}^{511}$:
$$\cos(\hat{u}, \hat{v}) = \hat{u} \cdot \hat{v}$$
Because $\|\hat{u}\|_2 = \|\hat{v}\|_2 = 1$, the dot product *equals* cosine similarity — making retrieval extremely fast (one matrix multiply).

**General Projection Head** $f_\theta: \mathbb{R}^{d_{\text{in}}} \to \mathbb{S}^{511}$:
$$h_1 = W_1 x + b_1, \quad h_2 = \text{GELU}(h_1), \quad h_3 = \text{LN}(h_2)$$
$$h_4 = W_2 h_3 + b_2, \quad h_5 = \text{LN}(h_4), \quad \hat{e} = h_5 / \|h_5\|_2$$

In [ ]:
import math, os, random, re, time, collections, pickle, copy
from collections import defaultdict, Counter
from typing import List, Tuple, Dict, Optional

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EMBED_DIM = 512        # Shared embedding dimension d

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'Shared embedding space: S^{EMBED_DIM-1} ⊂ R^{EMBED_DIM}')

## 1. General Projection Head

### Technical description
All modality encoders share the same MLP architecture to project into $\mathbb{S}^{511}$.

**Definition (Projection MLP):**
$$f_\theta(x) = \frac{\text{LN}(W_2 \cdot \text{GELU}(\text{LN}(W_1 x + b_1)) + b_2)}{\|\text{LN}(\cdots)\|_2}$$

**GELU activation:**
$$\text{GELU}(x) = x \cdot \Phi(x) \approx 0.5x\left(1 + \tanh\!\left[\sqrt{\tfrac{2}{\pi}}(x + 0.044715x^3)\right]\right)$$

**Layer Normalisation:** For $x \in \mathbb{R}^d$, $\mu = \frac{1}{d}\sum_i x_i$, $\sigma^2 = \frac{1}{d}\sum_i(x_i-\mu)^2$:
$$\text{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta$$

**Lemma (GELU vs ReLU):** GELU is smooth and non-zero for all $x$, providing continuous gradients unlike $\text{ReLU}(x)=\max(0,x)$ which has zero gradient for $x<0$ (dead neuron problem).

### Plain language
The projection head is a "translator" — it takes a large feature vector (e.g., 768 numbers from BERT or 2048 from ResNet) and squeezes it into a 512-number direction on an abstract sphere. The L2-normalisation at the end ensures every point sits exactly on the surface of this sphere, so comparing any two points is just a dot product.

In [ ]:
class ProjectionHead(nn.Module):
    """General projection head: R^{d_in} -> S^{511}.
    
    Architecture (from thesis Definition 3.1):
      Linear(d_in -> d_in) -> GELU -> LN(d_in) -> Linear(d_in -> 512) -> LN(512) -> L2-norm
    """
    def __init__(self, d_in: int, d_out: int = 512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_in),
            nn.GELU(),
            nn.LayerNorm(d_in),
            nn.Linear(d_in, d_out),
            nn.LayerNorm(d_out),
        )
        # Xavier uniform initialisation
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.net(x)
        return F.normalize(h, p=2, dim=-1)  # L2-normalise -> S^{d-1}


# ── Smoke test ─────────────────────────────────────────────────
proj_text  = ProjectionHead(d_in=768,  d_out=512)
proj_image = ProjectionHead(d_in=2048, d_out=512)
proj_audio = ProjectionHead(d_in=768,  d_out=512)

dummy_text  = torch.randn(4, 768)
dummy_image = torch.randn(4, 2048)
dummy_audio = torch.randn(4, 768)

e_text  = proj_text(dummy_text)
e_image = proj_image(dummy_image)
e_audio = proj_audio(dummy_audio)

print('Text  embedding shape :', e_text.shape,  '| L2-norm:', e_text.norm(dim=-1).mean().item())
print('Image embedding shape :', e_image.shape, '| L2-norm:', e_image.norm(dim=-1).mean().item())
print('Audio embedding shape :', e_audio.shape, '| L2-norm:', e_audio.norm(dim=-1).mean().item())
assert abs(e_text.norm(dim=-1).mean().item() - 1.0) < 1e-5, 'L2-norm must be 1.0'

## Algorithm 1 — Joint Multimodal Representation

### Technical description

**Definition (Joint Representation):**
Given modalities $\mathcal{M}_1, \mathcal{M}_2$ with encoder functions $f_k: \mathcal{X}_k \to \mathbb{R}^{d_k}$, the joint representation is:
$$\mathbf{r}_{\text{joint}} = \phi(\mathbf{e}_1, \mathbf{e}_2, \ldots, \mathbf{e}_K), \qquad \mathbf{e}_k = f_k(x_k)$$
where $\phi$ is a pooling function (concatenation, mean, etc.).

**Theorem (Expressiveness of Concatenation):** Concatenation $\phi_{\text{cat}}(\mathbf{e}_1,\ldots,\mathbf{e}_K) = [\mathbf{e}_1;\ldots;\mathbf{e}_K]$ is the maximal-rank pooling. A linear head can learn arbitrary linear combinations of all modality features.

```
Flowchart — Joint Representation:

  Modality 1 ──► Encoder_1 ──► e_1 ─┐
                                      ├──► Concat/Pool ──► Classifier ──► ŷ
  Modality 2 ──► Encoder_2 ──► e_2 ─┘
```

### Plain language
Joint representation is like stapling two experts' reports together before a judge reads them. The judge (classifier) sees everything at once. Simple, but requires all modalities at both training and inference time.

In [ ]:
# Algorithm 1: Joint Multimodal Representation
# ------------------------------------------------
# Input:  modality raw inputs {x_1,...,x_K}, encoders {f_k}, pooling phi, head g_theta
# Output: prediction y_hat, joint representation r

class JointMultimodalModel(nn.Module):
    """Concatenation-based joint multimodal representation.
    
    r_joint = [e_1 ; e_2 ; ... ; e_K]  (concatenation)
    y_hat   = g_theta(r_joint)
    """
    def __init__(self, embed_dims: List[int], num_classes: int, pooling: str = 'concat'):
        super().__init__()
        self.pooling = pooling
        if pooling == 'concat':
            joint_dim = sum(embed_dims)
        else:  # mean — requires all dims equal
            assert len(set(embed_dims)) == 1
            joint_dim = embed_dims[0]
        # Downstream classification head g_theta
        self.head = nn.Sequential(
            nn.Linear(joint_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, embeddings: List[torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor]:
        # Step 1: Pool representations
        if self.pooling == 'concat':
            r_joint = torch.cat(embeddings, dim=-1)
        else:
            r_joint = torch.stack(embeddings, dim=0).mean(dim=0)
        # Step 2: Downstream task
        y_hat = self.head(r_joint)
        return y_hat, r_joint


def run_joint_repr_demo():
    """Demo: two 512-d modality embeddings concatenated -> 1024-d -> 10-class output."""
    model = JointMultimodalModel(embed_dims=[512, 512], num_classes=10, pooling='concat')
    e1 = F.normalize(torch.randn(8, 512), dim=-1)
    e2 = F.normalize(torch.randn(8, 512), dim=-1)
    y_hat, r = model([e1, e2])
    print(f'[Alg 1] Joint repr | r_joint shape: {r.shape} | logits shape: {y_hat.shape}')

    # Visualise the joint space via PCA
    r_np = r.detach().numpy()
    from numpy.linalg import svd
    U, S, Vt = svd(r_np - r_np.mean(0), full_matrices=False)
    r_2d = U[:, :2] * S[:2]

    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    ax.scatter(r_2d[:, 0], r_2d[:, 1], c='steelblue', s=80, edgecolors='k', zorder=3)
    for i, (xx, yy) in enumerate(r_2d):
        ax.text(xx + .01, yy + .01, f'$\\mathbf{{r}}_{{{i}}}$', fontsize=8)
    ax.set_title('Joint Representation — PCA projection to 2D', fontsize=11)
    ax.set_xlabel('PC-1'); ax.set_ylabel('PC-2')
    ax.grid(True, alpha=.3)
    plt.tight_layout(); plt.show()

run_joint_repr_demo()

## Algorithm 2 — Coordinated Representation (Symmetric InfoNCE)

### Technical description

**Definition (Symmetric InfoNCE Loss):**
For a batch of $B$ positive pairs $\{(\hat{e}_A^{(i)}, \hat{e}_B^{(i)})\}_{i=1}^B$ with similarity matrix $S_{ij} = \hat{e}_A^{(i)} \cdot \hat{e}_B^{(j)} / \tau$:

$$\mathcal{L}_{A \to B} = -\frac{1}{B}\sum_{i=1}^B \log\frac{\exp(S_{ii})}{\sum_{j=1}^B \exp(S_{ij})}$$

$$\mathcal{L}_{\text{InfoNCE}} = \frac{1}{2}(\mathcal{L}_{A\to B} + \mathcal{L}_{B\to A}), \qquad \tau = 0.07$$

**Theorem (InfoNCE as MI Bound):**
$$I(f_A(x_A); f_B(x_B)) \geq \log B - \mathcal{L}_{\text{InfoNCE}}$$
Minimising $\mathcal{L}_{\text{InfoNCE}}$ maximises a lower bound on mutual information between the two modality representations.

**Lemma (Random Baseline):** At random initialisation $\mathbb{E}[\mathcal{L}_{\text{InfoNCE}}] = \log B$. For $B=32$: $\log 32 \approx 3.47$.

```
Similarity matrix S (B×B) — diagonal = positive pairs:

        ê_B^(1)  ê_B^(2)  ê_B^(3)
ê_A^(1) [HIGH]    low      low     <- softmax over row -> loss
ê_A^(2)  low     [HIGH]    low
ê_A^(3)  low      low     [HIGH]
```

### Plain language
InfoNCE is like a matching game. Show the model 32 images and 32 captions — all shuffled. The model must correctly pair each image with its caption. Loss = 0 means perfect pairing. Loss = 3.47 means pure random guessing. The temperature $\tau = 0.07$ makes the grader very strict — only very close matches score well.

In [ ]:
# Algorithm 2: Coordinated Representation — Symmetric InfoNCE
# -----------------------------------------------------------
# Input:  paired dataset D, frozen encoders f_A/f_B, trainable projection heads P_A/P_B
# Output: trained P_A*, P_B* that embed both modalities into shared S^{511}

def symmetric_infonce_loss(e_A: torch.Tensor, e_B: torch.Tensor,
                           tau: float = 0.07) -> torch.Tensor:
    """
    Symmetric InfoNCE (CLIP-style) contrastive loss.
    
    Args:
        e_A: L2-normalised embeddings from modality A, shape [B, d]
        e_B: L2-normalised embeddings from modality B, shape [B, d]
        tau: temperature (default 0.07 as in thesis)
    Returns:
        scalar loss
    """
    B = e_A.shape[0]
    # Ensure unit sphere
    e_A = F.normalize(e_A, p=2, dim=-1)
    e_B = F.normalize(e_B, p=2, dim=-1)
    # Similarity matrix S_{ij} = e_A^(i) . e_B^(j) / tau  [B x B]
    S = (e_A @ e_B.T) / tau
    labels = torch.arange(B, device=e_A.device)  # diagonal targets
    loss_A2B = F.cross_entropy(S,   labels)  # L_{A->B}
    loss_B2A = F.cross_entropy(S.T, labels)  # L_{B->A}
    return 0.5 * (loss_A2B + loss_B2A)


class CoordinatedRepresentationTrainer:
    """Trains two projection heads under symmetric InfoNCE."""
    def __init__(self, d_A: int, d_B: int, embed_dim: int = 512, tau: float = 0.07,
                 lr: float = 1e-4, device: torch.device = DEVICE):
        self.tau = tau
        self.proj_A = ProjectionHead(d_A, embed_dim).to(device)
        self.proj_B = ProjectionHead(d_B, embed_dim).to(device)
        self.optimizer = torch.optim.AdamW(
            list(self.proj_A.parameters()) + list(self.proj_B.parameters()),
            lr=lr, weight_decay=0.01
        )
        self.device = device

    def train_step(self, x_A: torch.Tensor, x_B: torch.Tensor) -> float:
        """Single gradient step (Algorithm 2, Steps 1-5)."""
        x_A, x_B = x_A.to(self.device), x_B.to(self.device)
        # Step 1-2: Encode and project
        e_A = self.proj_A(x_A)  # [B, 512] on S^{511}
        e_B = self.proj_B(x_B)  # [B, 512] on S^{511}
        # Step 3-4: Symmetric InfoNCE loss
        loss = symmetric_infonce_loss(e_A, e_B, self.tau)
        # Step 5: Update
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(self.proj_A.parameters()) + list(self.proj_B.parameters()), max_norm=1.0
        )
        self.optimizer.step()
        return loss.item()


def demo_infonce_convergence(n_epochs: int = 40, B: int = 32):
    """Synthetic demo: two matched 768-d modalities trained to align."""
    trainer = CoordinatedRepresentationTrainer(768, 768, device=DEVICE)
    losses = []
    random_baseline = math.log(B)

    for epoch in range(n_epochs):
        # Synthetic paired data: x_B = x_A + small noise (perfectly matchable)
        x_A = torch.randn(B, 768)
        x_B = x_A + 0.1 * torch.randn(B, 768)
        loss_val = trainer.train_step(x_A, x_B)
        losses.append(loss_val)
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d}/{n_epochs} | InfoNCE loss: {loss_val:.4f} '
                  f'(random baseline: {random_baseline:.2f})')

    # Plot convergence
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(losses, 'b-', linewidth=2, label='InfoNCE loss')
    ax.axhline(random_baseline, color='red', linestyle='--', label=f'Random baseline = log({B}) ≈ {random_baseline:.2f}')
    ax.set_xlabel('Training step'); ax.set_ylabel('Loss')
    ax.set_title('Algorithm 2 — Coordinated Representation (InfoNCE Convergence)')
    ax.legend(); ax.grid(True, alpha=.3)
    plt.tight_layout(); plt.show()

demo_infonce_convergence()

## Algorithm 3 — Missing-Modality Robust Inference

### Technical description

**Axiom (Zero-Vector Missing Modality):**
If modality $\mathcal{M}_k$ is absent, substituting $\mathbf{e}_k = \mathbf{0} \in \mathbb{R}^d$ in a linear projection layer results in zero contribution: $W_k \mathbf{0} = \mathbf{0}$. This provides **graceful degradation** without injecting noise.

**Theorem (Graceful Degradation):** A $K$-modality fusion model operating on $|\mathcal{A}| = K - m$ available modalities ($m$ missing) achieves performance at least as good as a $(K-m)$-modality model trained from scratch, if the shared embedding space is sufficiently regularised via contrastive learning.

```
Available:   [e_1,   e_2,   e_3]  -> normal fusion
e_2 missing: [e_1, ZERO,  e_3]  -> graceful degradation (no noise injection)
```

### Plain language
If a sensor is broken, instead of crashing the system, we substitute a zero vector — the neural network then simply ignores that channel without any noisy hallucination. Like having a panel of 3 judges where one is absent: the remaining two still vote, and the absent judge contributes nothing (not a random vote).

In [ ]:
# Algorithm 3: Missing-Modality Robust Inference
# -----------------------------------------------
# Input:  available modalities A ⊆ {1,...,K}, missing M = {1,...,K}\A
# Output: robust representation r

def missing_modality_robust_inference(
    embeddings: Dict[str, Optional[torch.Tensor]],
    modality_keys: List[str],
    embed_dim: int = 512,
    pooling: str = 'mean'
) -> torch.Tensor:
    """
    Algorithm 3: graceful degradation via zero-vector substitution.
    
    Args:
        embeddings : dict {modality_name -> tensor or None}
        modality_keys: ordered list of modality names
        embed_dim  : shared embedding dimension
        pooling    : 'mean' or 'concat'
    Returns:
        robust joint representation r
    """
    filled = []
    available = []
    for key in modality_keys:
        e = embeddings.get(key)
        if e is not None:
            filled.append(F.normalize(e, dim=-1))  # unit sphere
            available.append(key)
        else:
            B = next((v for v in embeddings.values() if v is not None), None)
            B_size = B.shape[0] if B is not None else 1
            zero = torch.zeros(B_size, embed_dim)
            filled.append(zero)  # zero-vector: W * 0 = 0 (Axiom)

    if not available:
        raise ValueError('All modalities missing — cannot produce a representation')

    stacked = torch.stack(filled, dim=1)         # [B, K, d]
    if pooling == 'mean':
        # Average only over available modalities
        mask = torch.tensor([1.0 if k in available else 0.0 for k in modality_keys])
        mask = mask.view(1, -1, 1)
        r = (stacked * mask).sum(dim=1) / mask.sum()
    else:
        r = stacked.view(stacked.shape[0], -1)   # concat: [B, K*d]
    return r


# Demo
B = 4
full_embeds  = {'text': torch.randn(B, 512), 'image': torch.randn(B, 512), 'audio': torch.randn(B, 512)}
miss_embeds  = {'text': torch.randn(B, 512), 'image': None,               'audio': torch.randn(B, 512)}
keys = ['text', 'image', 'audio']

r_full    = missing_modality_robust_inference(full_embeds, keys)
r_missing = missing_modality_robust_inference(miss_embeds, keys)

print(f'[Alg 3] All modalities present  | r shape: {r_full.shape}    | norm: {r_full.norm(dim=-1).mean().item():.3f}')
print(f'[Alg 3] Image modality MISSING  | r shape: {r_missing.shape} | norm: {r_missing.norm(dim=-1).mean().item():.3f}')
print('Missing modality handled gracefully — no crash, no noise injection.')

## Algorithm 4 — Model-Based Fusion (Cross-Modal Transformer)

### Technical description

**Theorem (Scaled Dot-Product Attention):**
$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$
The $1/\sqrt{d_k}$ factor prevents dot products from growing large in magnitude, which would saturate softmax and kill gradients.

**Multi-Head Attention:**
$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1,\ldots,\text{head}_h)W^O, \quad \text{head}_i = \text{Attn}(QW_i^Q, KW_i^K, VW_i^V)$$

**Pre-LN (Axiom — Stability):** In Pre-LN architecture:
$$x' = x + \text{Attn}(\text{LN}(x)), \qquad x'' = x' + \text{FFN}(\text{LN}(x'))$$
Pre-LN is more stable during early training than Post-LN because the residual path preserves gradient magnitude.

**Xavier Initialisation:**
$$W \sim \mathcal{U}\!\left(-\sqrt{\frac{6}{n_{\text{in}}+n_{\text{out}}}}, \sqrt{\frac{6}{n_{\text{in}}+n_{\text{out}}}}\right)$$

```
Flowchart — Model-Based Fusion:

  ê_text  ──►─────────────────────────────────────────►──┐
  ê_audio ──►── Stack + Pos.Enc ──► Transformer x2 ──►──┤ Mean-pool ──► Head ──► ŷ
  ê_video ──►─────────────────────────────────────────►──┘
               [B, 3, 512]        [B, 3, 512]
```

### Plain language
The Transformer fusion is like a committee meeting. Each modality (text, audio, video) sends a "delegate" (embedding). In the meeting room (attention layers), each delegate reads the other delegates' reports and updates their own view. After 2 rounds of discussion, all views are averaged into a final decision.

In [ ]:
# Algorithm 4: Model-Based Multimodal Fusion — Cross-Modal Transformer
# ---------------------------------------------------------------------
# Input:  modality embeddings {ê_k} ∈ S^{511}, parameters θ
# Output: prediction ŷ ∈ R

class MultimodalFusionTransformer(nn.Module):
    """
    Pipeline E fusion model (thesis §10, Algorithm 20).
    
    Architecture:
      Stack K modality tokens + learnable positional encodings
      -> L=2 Pre-LN Transformer encoder layers (h=8 heads)
      -> Mean-pool over K tokens
      -> Regression head: 512 -> 256 -> 1
    """
    def __init__(self, embed_dim: int = 512, num_modalities: int = 3,
                 num_layers: int = 2, num_heads: int = 8, ff_dim: int = 1024,
                 dropout: float = 0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_modalities = num_modalities

        # Learnable modality positional encodings P ∈ R^{K x d}
        self.pos_enc = nn.Parameter(torch.randn(num_modalities, embed_dim) * 0.02)

        # Pre-LN Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, norm_first=True  # Pre-LN
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Regression head
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)  # Xavier uniform
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, *embeddings: torch.Tensor) -> torch.Tensor:
        """
        Args:
            *embeddings: K tensors each of shape [B, 512], L2-normalised
        Returns:
            y_hat: [B] raw sentiment logit
        """
        # Step 1: Stack modality tokens -> [B, K, 512]
        X = torch.stack(list(embeddings), dim=1)
        # Step 2: Add learnable positional (modality) encodings
        X = X + self.pos_enc.unsqueeze(0)
        # Step 3: Cross-modal attention (Pre-LN, 2 layers)
        X = self.transformer(X)            # [B, K, 512]
        # Step 4: Mean-pool over modality tokens
        x_bar = X.mean(dim=1)              # [B, 512]
        # Step 5: Regression head
        y_hat = self.head(x_bar).squeeze(-1)  # [B]
        return y_hat


# Test
fusion_model = MultimodalFusionTransformer().to(DEVICE)
e_t = F.normalize(torch.randn(8, 512).to(DEVICE), dim=-1)
e_a = F.normalize(torch.randn(8, 512).to(DEVICE), dim=-1)
e_v = F.normalize(torch.randn(8, 512).to(DEVICE), dim=-1)
y_hat = fusion_model(e_t, e_a, e_v)
n_params = sum(p.numel() for p in fusion_model.parameters())
print(f'[Alg 4] Fusion Transformer | output shape: {y_hat.shape} | params: {n_params:,}')

## Algorithms 5 & 6 — Early Fusion and Late Fusion

### Technical description

**Definition (Early Fusion):**
$$\hat{y} = g_\theta(\phi_{\text{early}}(x_1, x_2, \ldots, x_K))$$
Features are merged *before* any learned model. The model $g_\theta$ sees only the fused feature, unaware of modality origin.

**Definition (Late Fusion):** $K$ independent models each produce a prediction $\hat{y}_k = g_k(x_k)$:
$$\hat{y} = \phi_{\text{late}}(\hat{y}_1, \hat{y}_2, \ldots, \hat{y}_K) = \sum_{k=1}^K w_k \hat{y}_k, \quad \sum_k w_k = 1$$

**Lemma (Early Fusion Trade-off):** Early fusion achieves lowest inference latency (single pass through $g_\theta$) but requires all modalities and cannot handle missing inputs.

**Proposition (Late Fusion Robustness):** Late fusion is the most robust to missing modalities — if $\mathcal{M}_k$ is absent, set $w_k = 0$ and re-normalise remaining weights.

### Plain language
- **Early fusion**: Mix all ingredients before cooking. Simple but inflexible.
- **Late fusion**: Cook each ingredient separately, then plate them together. More flexible, handles missing ingredients better.

In [ ]:
# Algorithm 5: Early Fusion
# -------------------------
class EarlyFusionModel(nn.Module):
    """Concatenate embeddings before the main model."""
    def __init__(self, modality_dims: List[int], num_classes: int):
        super().__init__()
        fused_dim = sum(modality_dims)
        self.model = nn.Sequential(
            nn.Linear(fused_dim, 512), nn.ReLU(),
            nn.Linear(512, 256),      nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, embeddings: List[torch.Tensor]) -> torch.Tensor:
        z = torch.cat(embeddings, dim=-1)   # Early fusion: concat before model
        return self.model(z)


# Algorithm 6: Late Fusion
# ------------------------
class LateFusionModel(nn.Module):
    """Independent per-modality models whose predictions are combined."""
    def __init__(self, modality_dims: List[int], num_classes: int,
                 weights: Optional[List[float]] = None):
        super().__init__()
        self.models = nn.ModuleList([
            nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, num_classes))
            for d in modality_dims
        ])
        if weights is None:
            weights = [1.0 / len(modality_dims)] * len(modality_dims)
        self.register_buffer('weights', torch.tensor(weights))

    def forward(self, embeddings: List[Optional[torch.Tensor]]) -> torch.Tensor:
        preds, active_w = [], []
        for i, (e, m) in enumerate(zip(embeddings, self.models)):
            if e is not None:                         # Skip missing modality
                preds.append(m(e))
                active_w.append(self.weights[i])
        w = torch.stack(active_w)
        w = w / w.sum()                               # Re-normalise
        return sum(w_i * p for w_i, p in zip(w, preds))


# Comparison demo
B, C = 16, 4
e1 = torch.randn(B, 512)
e2 = torch.randn(B, 512)

early = EarlyFusionModel([512, 512], C)
late  = LateFusionModel([512, 512],  C)

y_early = early([e1, e2])
y_late  = late([e1, e2])
y_late_missing = late([e1, None])   # missing e2

print(f'[Alg 5] Early fusion  | output: {y_early.shape}')
print(f'[Alg 6] Late  fusion  | output: {y_late.shape}')
print(f'[Alg 6] Late (miss e2)| output: {y_late_missing.shape}  <- graceful missing-mod handling')

## Algorithm 7 — TextEncoderV2 (BERT + TextProjection + NER)

### Technical description

**Text encoding pipeline:**
$$\text{caption} \xrightarrow{\text{WordPiece}} [t_1,\ldots,t_T] \xrightarrow{\text{BERT}} H \in \mathbb{R}^{T\times768} \xrightarrow{\text{mean-pool}} h_{\text{BERT}} \in \mathbb{R}^{768} \xrightarrow{P_\theta} \hat{e}_q \in \mathbb{S}^{511}$$

**Remark (Mean Pooling vs. CLS):**
- CLS token: $e = H_{:,0,:}$ — optimised for classification, degrades for long sequences.
- Mean pooling: $e = \frac{1}{T}\sum_{t=1}^T H_{:,t,:}$ — averages all token representations, better for semantic similarity (consistent with sentence-transformers literature).

**Domain Detection:**
$$n_m = |\{w \in \text{MEDICAL\_TERMS} : w \in t.\text{lower}()\}|, \qquad \text{hint} = \text{``medical'' if } n_m \geq 2$$

### Plain language
BERT reads the question and produces 768 numbers for each word. We average all those numbers to get one 768-number 'meaning vector' for the whole sentence. Then TextProjection squeezes it from 768 down to 512 while preserving meaning. Finally, we scale it to length exactly 1, placing it on the shared sphere. We also run a quick keyword scan to detect if the question is about medical topics.

In [ ]:
# Algorithm 7: TextEncoderV2 — BERT Mean-Pool + TextProjection + NER Domain Detection
# ------------------------------------------------------------------------------------
# Input:  text t, frozen BERT, trained TextProjection P_theta
# Output: e_q ∈ S^{511}, entity set E_q, domain_hint

MEDICAL_TERMS = {
    'pneumonia','radiology','xray','x-ray','mri','ct','ultrasound','pathology',
    'diagnosis','lesion','tumor','tumour','carcinoma','fracture','consolidation',
    'pleural','effusion','pneumothorax','cardiomegaly','atelectasis','edema',
    'opacity','infiltrate','nodule','calcification','stenosis','occlusion'
}

def simple_tokenise(text: str) -> List[str]:
    """Lightweight tokeniser (proxy for WordPiece)."""
    return re.findall(r'\b\w+\b', text.lower())


def text_encoder_v2(
    text: str,
    proj_head: ProjectionHead,
    bert_output_dim: int = 768,
    max_length: int = 128
) -> Tuple[torch.Tensor, List[str], Optional[str]]:
    """
    Algorithm 7: TextEncoderV2.encode
    
    Simulates the BERT mean-pool + TextProjection pipeline.
    In production: replace `bert_encode` with HuggingFace BERT.
    """
    tokens = simple_tokenise(text)[:max_length]

    # ── Step 2: Simulate BERT forward (random for demo; replace with real BERT) ──
    # Each token gets a 768-d hidden state; we mean-pool
    torch.manual_seed(abs(hash(text)) % (2**31))  # deterministic for same text
    T = max(len(tokens), 1)
    H = torch.randn(1, T, bert_output_dim)          # [1, T, 768] — proxy for BERT
    h_bert = H.mean(dim=1)                           # [1, 768]  — mean pooling

    # ── Step 3: TextProjection -> L2 normalise ──────────────────────────────────
    e_q = proj_head(h_bert).squeeze(0)              # [512] on S^{511}

    # ── Step 4: NER entity extraction (noun chunks, simplified) ─────────────────
    stop_words = {'a','an','the','is','of','in','on','at','to','and','or','for','with'}
    E_q = list(set(w for w in tokens if len(w) > 3 and w not in stop_words))

    # ── Step 5: Domain detection ─────────────────────────────────────────────────
    n_medical = sum(1 for w in tokens if w in MEDICAL_TERMS)
    domain_hint = 'medical' if n_medical >= 2 else None

    return e_q, E_q, domain_hint


# Test with general and medical queries
proj_text_v2 = ProjectionHead(d_in=768, d_out=512)

queries = [
    "a dog running in a park with green trees",
    "bilateral pleural effusion with pneumonia consolidation and cardiomegaly"
]

for q in queries:
    e, entities, hint = text_encoder_v2(q, proj_text_v2)
    print(f'Query : {q[:60]}...')
    print(f'  e_q  : shape={e.shape}, norm={e.norm().item():.4f}')
    print(f'  E_q  : {entities[:5]}')
    print(f'  hint : {hint}')
    print()

## Algorithm 8 — Pipeline A Training Epoch (InfoNCE, COCO)

### Technical description

**AdamW Update Rule:**
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$
$$\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\varepsilon} - \eta\lambda\theta_t$$

**Gradient Clipping:** $\tilde{\nabla}_\theta = \min\!\left(1, \frac{c}{\|\nabla_\theta\|_2}\right)\nabla_\theta$, $c=1.0$ — caps gradient norm without changing direction.

**Cosine Annealing LR Schedule:**
$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\frac{t\pi}{T}\right)$$

### Plain language
Each training step: (1) encode text and image into the shared sphere, (2) compute how well matched pairs are clustered vs. non-matched pairs (InfoNCE), (3) nudge the projection heads to do better using AdamW. Repeat for one epoch over all COCO batches. The learning rate starts high and gradually decreases (cosine annealing) — like easing your foot off the gas as you park.

In [ ]:
# Algorithm 8: Pipeline A Training Epoch (InfoNCE, COCO)
# -------------------------------------------------------
# Input:  COCO DataLoader, TextProjection P_theta, ImageProjection f_phi, tau=0.07
# Output: Updated P_theta, f_phi; epoch loss

class SyntheticCOCODataset(Dataset):
    """Synthetic proxy for COCO (image_feat, caption_feat) pairs."""
    def __init__(self, n=500, image_dim=2048, text_dim=768):
        self.image_feats = torch.randn(n, image_dim)
        # Caption is semantically related to image — add small noise
        base = torch.randn(n, text_dim)
        self.text_feats  = base + 0.05 * torch.randn(n, text_dim)
    def __len__(self): return len(self.image_feats)
    def __getitem__(self, idx): return self.image_feats[idx], self.text_feats[idx]


def train_one_epoch_pipeline_a(
    loader: DataLoader,
    proj_text: ProjectionHead,
    proj_image: ProjectionHead,
    optimizer: torch.optim.Optimizer,
    scheduler,
    tau: float = 0.07
) -> float:
    """Algorithm 8: single training epoch."""
    proj_text.train(); proj_image.train()
    total_loss = 0.0
    for image_feat, text_feat in loader:
        image_feat = image_feat.to(DEVICE)
        text_feat  = text_feat.to(DEVICE)
        # Step 1: Encode text (frozen BERT → proj head)
        e_T = proj_text(text_feat)     # [B, 512] on S^{511}
        # Step 2: Encode image (frozen ResNet → proj head)
        e_I = proj_image(image_feat)   # [B, 512] on S^{511}
        # Step 3: Symmetric InfoNCE loss
        loss = symmetric_infonce_loss(e_T, e_I, tau)
        # Step 4: Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(proj_text.parameters()) + list(proj_image.parameters()), max_norm=1.0
        )
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


# Run synthetic training
ds = SyntheticCOCODataset(n=256)
dl = DataLoader(ds, batch_size=32, shuffle=True, drop_last=True)

p_text  = ProjectionHead(768,  512).to(DEVICE)
p_image = ProjectionHead(2048, 512).to(DEVICE)
opt = torch.optim.AdamW(list(p_text.parameters()) + list(p_image.parameters()),
                        lr=1e-4, weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=len(dl)*10, eta_min=1e-6)

epoch_losses = []
for ep in range(10):
    loss_val = train_one_epoch_pipeline_a(dl, p_text, p_image, opt, sched)
    epoch_losses.append(loss_val)
    print(f'  Epoch {ep+1:2d}/10 | Loss: {loss_val:.4f}')

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(epoch_losses, 'o-', color='steelblue', linewidth=2)
ax.axhline(math.log(32), color='red', linestyle='--', label='Random baseline log(32)≈3.47')
ax.set_title('Algorithm 8 — Pipeline A Training (Synthetic COCO)')
ax.set_xlabel('Epoch'); ax.set_ylabel('InfoNCE Loss'); ax.legend(); ax.grid(True, alpha=.3)
plt.tight_layout(); plt.show()

## Algorithms 9 & 10 — PMI Knowledge Graph Construction and DFS Search

### Technical description

**Definition (PMI — Pointwise Mutual Information):**
For entities $a, b$ in a corpus of $N$ documents:
$$\text{PMI}(a, b) = \log\frac{P(a,b)}{P(a)\cdot P(b)} = \log\frac{c_{ab} \cdot N}{f_a \cdot f_b}$$
- $c_{ab}$: co-occurrence count; $f_a$, $f_b$: document frequencies
- $\text{PMI} > 0 \Leftrightarrow$ entities co-occur more than expected under independence

**Example:** $N=200\,000$, $f_{\text{dog}}=8000$, $f_{\text{leash}}=1200$, $c=900$:
$$\text{PMI}(\text{dog},\text{leash}) = \log\frac{900 \times 200000}{8000 \times 1200} = \log(18.75) \approx 2.93$$

**Algorithm 10 — 2-Hop DFS:**
Depth-first search over $\mathcal{G}$ limited to 2 hops, collecting all entity neighbours of a query entity.

### Plain language
The PMI KG is a semantic dictionary: it knows that 'dog' and 'leash' appear together far more often than chance, so they're connected. During retrieval, if you ask about 'dog', the KG boosts documents mentioning 'leash', 'collar', and 'park' — even if those words weren't in your query. The 2-hop DFS finds friends-of-friends in this semantic network.

In [ ]:
# Algorithms 9 & 10: PMI Knowledge Graph Construction and DFS Search
# -------------------------------------------------------------------

STOPWORDS = {
    'the','a','an','is','are','was','were','be','been','being',
    'have','has','had','do','does','did','will','would','shall',
    'should','may','might','must','can','could','in','on','at',
    'to','for','of','and','or','but','if','then','than','with'
}

def build_pmi_kg(
    captions: List[str],
    top_entities: int = 100,
    min_cooc: int = 2,
    min_word_len: int = 3
) -> Dict[str, Dict[str, float]]:
    """
    Algorithm 9: Build PMI-weighted Knowledge Graph.
    
    Returns:
        kg: {entity_a -> {entity_b -> PMI_score}}
    """
    N = len(captions)
    # Step 1: Collect all words
    all_words = []
    for cap in captions:
        all_words += re.findall(r'\b[a-z]+\b', cap.lower())

    counts = Counter(all_words)
    # Step 2: Top-V entities (non-stopword, min length)
    vocab = {w for w, c in counts.most_common(top_entities * 3)
             if w not in STOPWORDS and len(w) >= min_word_len}
    vocab = set(list(vocab)[:top_entities])

    # Step 3: Build co-occurrence matrix
    cooc: Dict[str, Counter] = defaultdict(Counter)
    doc_freq: Counter = Counter()
    for cap in captions:
        ents = set(w for w in re.findall(r'\b[a-z]+\b', cap.lower()) if w in vocab)
        for e in ents:
            doc_freq[e] += 1
        for e1 in ents:
            for e2 in ents:
                if e1 != e2:
                    cooc[e1][e2] += 1

    # Step 4: Compute PMI and build KG
    kg: Dict[str, Dict[str, float]] = defaultdict(dict)
    for a, neighbours in cooc.items():
        for b, c_ab in neighbours.items():
            if c_ab < min_cooc:
                continue
            f_a = max(doc_freq[a], 1)
            f_b = max(doc_freq[b], 1)
            pmi = math.log(c_ab * N / (f_a * f_b + 1e-9) + 1e-9)
            if pmi > 0:
                kg[a][b] = round(pmi, 4)
    return dict(kg)


def kg_dfs_neighbours(
    entity: str, kg: Dict[str, Dict[str, float]], max_hops: int = 2
) -> List[Tuple[str, str, str, int]]:
    """
    Algorithm 10: Depth-First Neighbour Search (max 2 hops).
    Returns list of (head, relation, tail, hop_count) tuples.
    """
    visited, neighbours = set(), []

    def dfs(current: str, hop: int):
        if hop > max_hops or current in visited:
            return
        visited.add(current)
        for neighbour, pmi in kg.get(current, {}).items():
            neighbours.append((current, 'appears_with', neighbour, hop))
            dfs(neighbour, hop + 1)

    dfs(entity, 1)
    return neighbours


# Build KG from synthetic COCO-like captions
synthetic_captions = [
    "a dog runs through the park with a leash",
    "two dogs play fetch with a ball in the park",
    "a cat sits on the mat near the window",
    "the bicycle is parked next to the tree",
    "people walk their dogs in the sunny park",
    "a dog catches a ball near the bench",
    "the window overlooks the garden with trees",
    "children play football in the park",
    "the cat watches birds through the window",
    "a red bicycle leans against the fence",
]
kg = build_pmi_kg(synthetic_captions, top_entities=50)

print(f'KG entities: {len(kg)}')
print('Top PMI pairs for "dog":', sorted(kg.get('dog', {}).items(), key=lambda x: -x[1])[:5])

neighbours = kg_dfs_neighbours('dog', kg, max_hops=2)
print(f'\n2-hop DFS from "dog" | found {len(neighbours)} neighbour triples:')
for h, r, t, hop in neighbours[:8]:
    print(f'  hop={hop} | {h} --[{r}]--> {t} (PMI={kg.get(h,{}).get(t,0):.3f})')

## Algorithm 11 — Pipeline B: Surgical Domain Transfer (ROCO)

### Technical description

**Theorem (Surgical Transfer Efficiency):**
$$\eta_{\text{surgical}} = \frac{|\phi|}{|\phi_{\text{total}}|} = \frac{5.25}{235.15} \approx 2.23\%$$
Only ImageProjection (5.25M) is retrained; all 229.9M other parameters are frozen.

**Dual Loss with Knowledge Distillation:**
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{align}} + \lambda(t) \cdot \mathcal{L}_{\text{distill}}$$
$$\mathcal{L}_{\text{distill}} = \frac{1}{B}\sum_{i=1}^B \left\|f_T^{\text{ROCO}}(c_i) - f_T^{\text{M2}}(c_i)\right\|_2^2$$

**Lemma (Domain Swap Isolation):** Replacing $P_I^{\text{src}} \to P_I^{\text{tgt}}$ does not move existing text/audio embeddings in $\mathbb{S}^{511}$, so ROCO transfer cannot degrade COCO text retrieval.

### Plain language
Like teaching a student who already speaks English (COCO) to understand X-ray reports (ROCO). We freeze their English skills and only retrain the 'visual translator' — the thin layer connecting 'what the image looks like' to 'what it means in the shared semantic space'. The distillation loss is a safety net: don't forget your English while learning radiology.

In [ ]:
# Algorithm 11: Pipeline B — Surgical Domain Transfer (ROCO)
# ----------------------------------------------------------
# Input:  Source frozen TextProjection, target ROCO dataset, fresh ImageProjection
# Output: Trained image_projection_roco.pth

class SyntheticROCODataset(Dataset):
    """Proxy for ROCO (X-ray pool5 features, caption text features)."""
    def __init__(self, n=200, img_dim=2048, text_dim=768):
        self.img_feats  = torch.randn(n, img_dim)
        base = torch.randn(n, text_dim)
        self.text_feats = base + 0.1 * torch.randn(n, text_dim)
    def __len__(self): return len(self.img_feats)
    def __getitem__(self, i): return self.img_feats[i], self.text_feats[i]


def finetune_image_projection_roco(
    loader: DataLoader,
    frozen_text_proj: ProjectionHead,   # pre-trained Month-2 TextProjection
    n_epochs: int = 5,
    lr_img: float = 5e-4,
    lr_txt: float = 1e-5,
    lam: float = 0.5,
    tau: float = 0.07
) -> Tuple[ProjectionHead, List[float]]:
    """
    Algorithm 11: Surgical domain transfer.
    - ImageProjection: fully trained (lr=5e-4)
    - TextProjection:  minor nudge    (lr=1e-5, optional)
    """
    img_proj  = ProjectionHead(2048, 512).to(DEVICE)
    txt_proj  = copy.deepcopy(frozen_text_proj).to(DEVICE)  # start from Month-2 weights

    opt_img = torch.optim.AdamW(img_proj.parameters(),  lr=lr_img, weight_decay=0.01)
    opt_txt = torch.optim.AdamW(txt_proj.parameters(),  lr=lr_txt, weight_decay=0.01)

    losses = []
    for epoch in range(n_epochs):
        img_proj.train(); txt_proj.train()
        ep_loss = 0.0
        for img_feat, text_feat in loader:
            img_feat  = img_feat.to(DEVICE)
            text_feat = text_feat.to(DEVICE)

            # 1. Text embeddings (ROCO captions)
            e_T_roco = txt_proj(text_feat)            # trainable nudge
            # 2. Image embeddings (X-ray features)
            e_I_roco = img_proj(img_feat)

            # 3. Alignment loss (InfoNCE)
            L_align = symmetric_infonce_loss(e_T_roco, e_I_roco, tau)

            # 4. Distillation loss (preserve Month-2 text space)
            with torch.no_grad():
                e_T_frozen = frozen_text_proj(text_feat.to('cpu')).to(DEVICE)
            L_distill = F.mse_loss(e_T_roco, e_T_frozen)

            # 5. Combined loss with warm-up lambda
            L = L_align + lam * L_distill

            opt_img.zero_grad(); opt_txt.zero_grad()
            L.backward()
            torch.nn.utils.clip_grad_norm_(img_proj.parameters(), 1.0)
            opt_img.step(); opt_txt.step()
            ep_loss += L.item()

        avg = ep_loss / len(loader)
        losses.append(avg)
        print(f'  Epoch {epoch+1}/{n_epochs} | align+distill loss: {avg:.4f}')

    return img_proj, losses


# Demonstrate surgical transfer
frozen_tp = ProjectionHead(768, 512)   # pretend this is Month-2 TextProjection
roco_ds   = SyntheticROCODataset(n=128)
roco_dl   = DataLoader(roco_ds, batch_size=32, shuffle=True, drop_last=True)

print('=== Algorithm 11: Surgical Domain Transfer (ROCO) ===')
print(f'Total params: ImageProj={sum(p.numel() for p in ProjectionHead(2048,512).parameters()):,} '
      f'(≈2.23% of 235M total)')
img_proj_roco, losses_roco = finetune_image_projection_roco(roco_dl, frozen_tp)

## Algorithm 12 — NER KG Re-ranker (Two-Stage Hybrid Retrieval, Stage 2)

### Technical description

**Two-stage hybrid retrieval:**
$$\text{Stage 1: FAISS} \to \text{top-}K=20 \text{ shortlist} \quad \text{(dense cosine similarity)}$$
$$\text{Stage 2: KG PMI re-rank} \to \text{top-}N=5 \text{ final context}$$

**Hybrid Score:**
$$\tilde{s}_{\text{dense}}(d_i) = \frac{s_i - s_{\min}}{s_{\max} - s_{\min}}, \qquad \tilde{s}_{\text{KG}}(d_i) = \frac{\text{kg\_raw}(d_i)}{\max_j \text{kg\_raw}(d_j)}$$
$$\text{final\_score}(d_i) = \alpha \cdot \tilde{s}_{\text{dense}}(d_i) + (1-\alpha)\cdot\tilde{s}_{\text{KG}}(d_i), \quad \alpha = 0.7$$

**Theorem (Score Bounds):** $\text{final\_score}(d_i) \in [0, 1]$ for all $d_i$ by construction (min-max normalisation on each term).

### Plain language
Stage 1 is a fast keyword-free search (pure semantic similarity). Stage 2 adds a second opinion from the knowledge graph: documents that mention entities related to your query get a bonus. The final ranking combines both scores. Like a librarian who first finds books by topic (stage 1), then checks which ones also appear on a recommended reading list (stage 2).

In [ ]:
# Algorithm 12: NER KG Re-ranker (Hybrid Retrieval Stage 2)
# ----------------------------------------------------------
# Input:  shortlist S (top-K from FAISS), query entities E_q, KG, alpha=0.7
# Output: top-N documents with final_score ∈ [0,1]

def ner_kg_rerank(
    shortlist: List[Dict],          # [{text, dense_score}, ...] from FAISS top-K
    query_entities: List[str],
    kg: Dict[str, Dict[str, float]],
    top_n: int = 5,
    alpha: float = 0.7
) -> List[Dict]:
    """
    Algorithm 12: PMI entity re-ranking over FAISS shortlist.
    """
    if not shortlist:
        return []

    dense_scores = np.array([d['dense_score'] for d in shortlist])

    # Step 1: Min-max normalise dense scores
    s_min, s_max = dense_scores.min(), dense_scores.max()
    delta = max(s_max - s_min, 1e-8)
    s_dense_norm = (dense_scores - s_min) / delta   # ∈ [0, 1]

    # Step 2: PMI entity overlap scores
    kg_raw = np.zeros(len(shortlist))
    for i, doc in enumerate(shortlist):
        doc_words = set(re.findall(r'\b[a-z]+\b', doc['text'].lower()))
        for q_ent in query_entities:
            if q_ent in kg:
                for d_ent in doc_words:
                    if d_ent in kg[q_ent]:
                        kg_raw[i] += kg[q_ent][d_ent]

    # Step 3: Normalise KG scores
    m_kg = max(kg_raw.max(), 1e-8)
    s_kg_norm = kg_raw / m_kg                        # ∈ [0, 1]

    # Step 4: Combine
    final_scores = alpha * s_dense_norm + (1 - alpha) * s_kg_norm

    # Step 5: Sort and truncate
    ranked_idx = np.argsort(-final_scores)
    results = []
    for idx in ranked_idx[:top_n]:
        doc = dict(shortlist[idx])
        doc['final_score'] = float(final_scores[idx])
        doc['dense_norm']  = float(s_dense_norm[idx])
        doc['kg_norm']     = float(s_kg_norm[idx])
        results.append(doc)
    return results


# Demo with synthetic FAISS shortlist
shortlist_demo = [
    {'text': 'a dog runs through the park with a leash',   'dense_score': 0.85},
    {'text': 'cats and dogs playing outside',              'dense_score': 0.80},
    {'text': 'the park has trees benches and dogs',        'dense_score': 0.75},
    {'text': 'a bicycle leans against the park fence',     'dense_score': 0.70},
    {'text': 'children play football in the sunny park',   'dense_score': 0.65},
    {'text': 'a cat sits on the windowsill watching birds','dense_score': 0.60},
]
query_ents = ['dog', 'park', 'leash']
top_docs = ner_kg_rerank(shortlist_demo, query_ents, kg, top_n=3, alpha=0.7)

print('=== Algorithm 12: NER KG Re-ranker ===')
print(f'Query entities: {query_ents}\n')
for i, doc in enumerate(top_docs):
    print(f'  Rank {i+1}: final={doc["final_score"]:.3f} '
          f'(dense_norm={doc["dense_norm"]:.3f}, kg_norm={doc["kg_norm"]:.3f})')
    print(f'         "{doc["text"][:70]}"')

## Algorithm 13 — Full Query Pipeline (Tiered Confidence RAG)

### Technical description

**Definition (Confidence Tier Function):**
$$\text{tier}(s) = \begin{cases} \texttt{full} & s \geq \tau \\ \texttt{partial} & \tau - 0.2 \leq s < \tau \\ \texttt{rejected} & s < \tau - 0.2 \end{cases}, \quad \tau_{\text{general}} = 0.40, \;\tau_{\text{medical}} = 0.55$$

**Axiom (Clinical Safety Constraint):** For clinical domains, false positives carry higher risk than false negatives: $\tau_{\text{medical}} = \tau_{\text{general}} + 0.15$.

**Theorem (RAG Faithfulness):** An answer is faithful iff every claim $c$ in the generated answer can be inferred from some $d_j \in \mathcal{D}^*$. Enforced by: (1) "Answer using ONLY the retrieved context", (2) requiring citation, (3) rejecting low-confidence retrievals before any LLM call.

### Plain language
The full pipeline is like a cautious librarian: (1) understand the question, (2) detect if it's medical (higher safety bar), (3) search two stages for relevant documents, (4) decide confidence level (full/partial/rejected), (5) only then ask the language model to answer — and only from the found documents, never from imagination.

In [ ]:
# Algorithm 13: Full Query Pipeline (Tiered Confidence RAG)
# ---------------------------------------------------------

def confidence_tier(s_max: float, domain: str = 'general') -> str:
    """Classify retrieval confidence into full / partial / rejected."""
    tau = 0.55 if domain == 'medical' else 0.40
    if s_max >= tau:
        return 'full'
    elif s_max >= tau - 0.2:
        return 'partial'
    else:
        return 'rejected'


class MockRAGPipeline:
    """Simplified, self-contained RAG pipeline (no real LLM/FAISS needed)."""

    def __init__(self, corpus: List[Dict], kg: Dict, proj_head: ProjectionHead):
        self.corpus   = corpus
        self.kg       = kg
        self.proj     = proj_head
        self.domain   = 'general'
        # Build corpus embeddings (proxy)
        self.corpus_embs = [
            F.normalize(torch.randn(1, 512), dim=-1).squeeze(0)
            for _ in corpus
        ]

    def detect_domain(self, hint: Optional[str]) -> None:
        if hint == 'medical' and self.domain != 'medical':
            self.domain = 'medical'
            print('  [Domain swap] general -> medical (< 5s)')

    def faiss_top_k(self, e_q: torch.Tensor, k: int = 6) -> List[Dict]:
        """Simulate FAISS top-K retrieval."""
        sims = [float((e_q * emb).sum()) for emb in self.corpus_embs]
        ranked = sorted(zip(sims, self.corpus), key=lambda x: -x[0])
        return [{'text': d['text'], 'dense_score': float(s)} for s, d in ranked[:k]]

    def query(self, text: str) -> Dict:
        # Phase 1: Encode
        e_q, E_q, hint = text_encoder_v2(text, self.proj)
        # Phase 1b: Domain routing
        self.detect_domain(hint)
        # Phase 2: Retrieval
        shortlist = self.faiss_top_k(e_q, k=6)
        top_docs  = ner_kg_rerank(shortlist, E_q, self.kg, top_n=3)
        s_max     = max(d['final_score'] for d in top_docs) if top_docs else 0.0
        tier      = confidence_tier(s_max, self.domain)
        # Phase 3: Generate or reject
        if tier == 'rejected':
            answer = f"I cannot confidently answer (confidence={s_max:.3f} < threshold)."
        else:
            context = ' | '.join(d['text'] for d in top_docs[:3])
            answer = f"[{tier.upper()} confidence={s_max:.3f}] Based on: {context[:120]}..."
        return {'query': text, 'tier': tier, 's_max': s_max, 'answer': answer,
                'top_docs': top_docs, 'domain': self.domain}


corpus_demo = [{'text': c} for c in [
    "a dog runs through the park with a leash",
    "cats and dogs playing outside in the garden",
    "bilateral pleural effusion with pneumonia consolidation",
    "cardiomegaly and pulmonary edema on chest xray",
    "children play football in the sunny park",
]]
rag = MockRAGPipeline(corpus_demo, kg, ProjectionHead(768, 512))

print('=== Algorithm 13: Full Query Pipeline (Tiered Confidence RAG) ===')
test_queries = [
    "what kind of dog is running in the park",
    "pleural effusion cardiomegaly pneumonia chest findings",
]
for q in test_queries:
    result = rag.query(q)
    print(f'\nQuery: {q}')
    print(f'  Domain: {result["domain"]} | Tier: {result["tier"]} | s_max: {result["s_max"]:.3f}')
    print(f'  Answer: {result["answer"][:120]}')

## Algorithms 14–17 — Pipeline C: RAVDESS Audio Emotion (SupCon)

### Technical description

**Bug fixes from thesis:**
- Bug 1: Contaminated joint training. Fix: standalone audio-only training.
- Bug 2: `wav2vec_dim=512` instead of 768. Fix: use correct 768-d Wav2Vec output.

**Supervised Contrastive Loss (Alg. 15):**
$$\mathcal{P}(i) = \{p \in \{1,\ldots,B\} \setminus \{i\} : y_p = y_i\}, \quad A(i) = \{1,\ldots,B\} \setminus \{i\}$$
$$\mathcal{L}_{\text{SupCon}} = \frac{1}{B}\sum_{i=1}^B \frac{-1}{|\mathcal{P}(i)|}\sum_{p \in \mathcal{P}(i)} \log\frac{\exp(\hat{e}_i \cdot \hat{e}_p / \tau)}{\sum_{a \in A(i)} \exp(\hat{e}_i \cdot \hat{e}_a / \tau)}$$

**Lemma (Temperature effect):** $\|\partial\mathcal{L}/\partial S_{ij}\| \propto 1/\tau$. At $\tau=0.07$, gradients are $\approx 14\times$ larger than at $\tau=1.0$, producing tighter emotion clusters.

**Spherical Centroid (Alg. 17):**
$$\bar{c}_e = \frac{\mu_e}{\|\mu_e\|_2}, \qquad \mu_e = \frac{1}{N_e}\sum_{i=1}^{N_e}\hat{a}_i^{(e)}$$
Voronoi classifier: $\hat{y} = \arg\max_c \hat{e} \cdot \bar{c}_c$

### Plain language
SupCon is a team-building game: each audio clip looks at all other clips in the batch. Same-emotion clips are teammates — pulled closer together on the sphere. Different-emotion clips are opponents — pushed further apart. Temperature $\tau=0.07$ makes the grader very strict. After training, we compute the 'average direction' for each emotion — the centroid. At inference, we classify by finding the nearest centroid.

In [ ]:
# Algorithms 14-17: Pipeline C — RAVDESS SupCon Audio Emotion
# -----------------------------------------------------------

RAVDESS_EMOTIONS = ['neutral','calm','happy','sad','angry','fearful','disgust','surprised']
EMOTION_TO_IDX   = {e: i for i, e in enumerate(RAVDESS_EMOTIONS)}

# Algorithm 14: Embedding Cache (proxy — real version uses Wav2Vec)
class RAVDESSEmbeddingCache(Dataset):
    """Synthetic proxy for 768-d Wav2Vec embeddings of RAVDESS clips."""
    def __init__(self, n_per_class: int = 180, wav2vec_dim: int = 768):
        embs, labels = [], []
        for label in range(8):
            base = torch.randn(1, wav2vec_dim)  # emotion 'centroid'
            embs.append(base.repeat(n_per_class, 1) + 0.5 * torch.randn(n_per_class, wav2vec_dim))
            labels += [label] * n_per_class
        self.embeddings = torch.cat(embs, dim=0)  # [N, 768]
        self.labels     = torch.tensor(labels)    # [N]

    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.embeddings[i], self.labels[i]


# Algorithm 15: SupCon Loss
def supcon_loss(embeddings: torch.Tensor, labels: torch.Tensor,
                tau: float = 0.07) -> torch.Tensor:
    """
    Supervised Contrastive Loss (Khosla et al., 2020).
    """
    B = embeddings.shape[0]
    e = F.normalize(embeddings, dim=-1)          # ensure unit sphere
    sim = (e @ e.T) / tau                        # [B, B] similarity matrix

    # Positive mask: same label, exclude self (diagonal)
    labels_col = labels.unsqueeze(1)
    pos_mask = (labels_col == labels_col.T).float()
    pos_mask.fill_diagonal_(0)                   # exclude self

    # Log-sum-exp numerical stability
    sim_max = sim.detach().max(dim=1, keepdim=True).values
    sim = sim - sim_max

    # Exclude self from denominator
    self_mask = torch.eye(B, device=embeddings.device)
    exp_sim   = torch.exp(sim) * (1 - self_mask)
    log_prob  = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-9)

    n_pos = pos_mask.sum(dim=1)                  # positives per anchor
    valid = (n_pos > 0)
    if not valid.any():
        return torch.tensor(0.0, device=embeddings.device)

    per_anchor = -(pos_mask * log_prob).sum(dim=1)
    return (per_anchor[valid] / n_pos[valid]).mean()


# Algorithm 16: SupCon Training Epoch
def supcon_train_epoch(loader, audio_proj, optimizer) -> float:
    audio_proj.train()
    total = 0.0
    for emb_batch, label_batch in loader:
        emb_batch   = emb_batch.to(DEVICE)
        label_batch = label_batch.to(DEVICE)
        projected   = audio_proj(emb_batch)           # [B, 512] on S^{511}
        loss        = supcon_loss(projected, label_batch)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(audio_proj.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


# Algorithm 17: Centroid Nearest-Neighbour Evaluation
@torch.no_grad()
def evaluate_centroid_nn(loader, audio_proj) -> Tuple[float, Dict[str,float], torch.Tensor]:
    audio_proj.eval()
    all_embs, all_labels = [], []
    for emb_batch, label_batch in loader:
        proj = audio_proj(emb_batch.to(DEVICE)).cpu()
        all_embs.append(proj); all_labels.append(label_batch)

    E = torch.cat(all_embs);  Y = torch.cat(all_labels)
    # Compute spherical centroids
    centroids = []
    for c in range(8):
        mask = (Y == c)
        if mask.any():
            mu = E[mask].mean(dim=0)
            centroids.append(F.normalize(mu.unsqueeze(0), dim=-1))
        else:
            centroids.append(torch.zeros(1, 512))
    C = torch.cat(centroids, dim=0)              # [8, 512]
    # Classify by nearest centroid (Voronoi)
    sims = E @ C.T                               # [N, 8]
    predicted = sims.argmax(dim=1)
    acc = (predicted == Y).float().mean().item()
    per_emo = {RAVDESS_EMOTIONS[c]: float((predicted[Y==c] == c).float().mean())
               for c in range(8) if (Y==c).any()}
    return acc, per_emo, C


# Run Pipeline C training
cache_ds = RAVDESSEmbeddingCache(n_per_class=45)
cache_dl = DataLoader(cache_ds, batch_size=16, shuffle=True, drop_last=True)

audio_proj = ProjectionHead(768, 512).to(DEVICE)
optimizer  = torch.optim.AdamW(audio_proj.parameters(), lr=1e-4, weight_decay=0.01)

print('=== Algorithms 15-17: Pipeline C — SupCon Emotion Training ===')
print(f'Random baseline: log(15) ≈ {math.log(15):.3f}')
sc_losses, sc_accs = [], []
for ep in range(15):
    l = supcon_train_epoch(cache_dl, audio_proj, optimizer)
    acc, per_emo, centroids = evaluate_centroid_nn(cache_dl, audio_proj)
    sc_losses.append(l); sc_accs.append(acc)
    if (ep+1) % 5 == 0:
        print(f'  Epoch {ep+1:2d}/15 | SupCon loss: {l:.4f} | Centroid-NN acc: {acc*100:.1f}%')

print(f'\nFinal per-emotion accuracy:')
for emo, a in per_emo.items():
    bar = '█' * int(a * 20)
    print(f'  {emo:12s}: {bar:<20} {a*100:.1f}%')

## Algorithms 18 & 19 — Pipeline D: CREMA-D Audio-Visual Emotion (3-Phase)

### Technical description

**Middle-Frame Extraction (Alg. 18):**
For a video of $N$ frames, the representative frame index is $f^* = \lfloor N/2 \rfloor$.

**Theorem (Peak Expression Frame):** The middle frame of a short acted-emotion clip ($\leq 3$s) statistically maximises the probability of capturing the peak facial action unit configuration.

**Cross-Modal InfoNCE (Stage 1, Alg. 19):**
$$S_{ij} = \hat{e}_i^{\text{audio}} \cdot \hat{e}_j^{\text{video}}, \qquad \mathcal{L}_{\text{InfoNCE}} = \frac{1}{2}\left[-\frac{1}{B}\sum_i\log\frac{e^{S_{ii}/\tau}}{\sum_j e^{S_{ij}/\tau}} - \frac{1}{B}\sum_j\log\frac{e^{S_{jj}/\tau}}{\sum_i e^{S_{ij}/\tau}}\right]$$

**Three-Phase Training Arc:**
| Phase | Loss | LR | Epochs | Result |
|---|---|---|---|---|
| 1 | Cross-modal InfoNCE | $10^{-4}$ | 13 | 42.2% (1/6) |
| 2 | SupCon | $10^{-5}$ | 10 | 43.9% (3/6) |
| 3 | SupCon | $5\times10^{-5}$ | 15 | Target $\geq$4/6 |

**Theorem (Phase 2 LR Insufficiency):** Weight update $\Delta_{\text{Phase 2}} \approx 10^{-5} \times 12.4 \approx 1.24 \times 10^{-4}$ (marginal). Phase 3 gives $5\times$ larger updates while preserving cross-modal geometry.

### Plain language
Pipeline D teaches the robot to recognise 6 emotions from video clips (CREMA-D). Phase 1 pairs each audio clip with its video (middle frame) — "match the voice to the face." Phase 2 and 3 fine-tune using emotion labels. The three-phase strategy is like learning to dance: first get the partner's rhythm (InfoNCE), then polish your footwork (SupCon), then add flair (SupCon with bigger steps).

In [ ]:
# Algorithms 18 & 19: Pipeline D — CREMA-D Audio-Visual Emotion
# -------------------------------------------------------------

CREMA_EMOTIONS = ['angry','disgust','fearful','happy','neutral','sad']

# Algorithm 18: Middle-Frame Extraction (proxy without OpenCV)
def extract_middle_frame_proxy(n_frames: int, frame_dim: int = 2048) -> torch.Tensor:
    """
    Proxy for cv2 middle-frame extraction.
    In production: cap.set(CAP_PROP_POS_FRAMES, max(0, N//2)); cap.read()
    Returns ResNet-50 pool5 features [2048] of the middle frame.
    """
    mid_idx = n_frames // 2
    # Simulate ImageNet-normalised frame -> ResNet GAP
    frame_feat = torch.randn(1, frame_dim)
    return frame_feat.squeeze(0)


class SyntheticCREMADDataset(Dataset):
    """Proxy CREMA-D with paired audio (768-d) and video (2048-d) features."""
    def __init__(self, n_per_class=60):
        audio_list, video_list, label_list = [], [], []
        for c in range(6):
            audio_base = torch.randn(1, 768)
            video_base = torch.randn(1, 2048)
            n = n_per_class
            audio_list.append(audio_base.repeat(n,1) + 0.5*torch.randn(n, 768))
            video_list.append(video_base.repeat(n,1) + 0.5*torch.randn(n, 2048))
            label_list += [c] * n
        self.audio = torch.cat(audio_list)
        self.video = torch.cat(video_list)
        self.labels = torch.tensor(label_list)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.audio[i], self.video[i], self.labels[i]


# Algorithm 19: Stage 1 — Cross-Modal InfoNCE Training
def stage1_infonce_epoch(loader, audio_proj, video_proj, optimizer, tau=0.07) -> float:
    """Cross-modal InfoNCE: align audio embeddings with video embeddings."""
    audio_proj.train(); video_proj.train()
    total = 0.0
    for audio_feat, video_feat, _ in loader:
        e_A = audio_proj(audio_feat.to(DEVICE))  # [B,512]
        e_V = video_proj(video_feat.to(DEVICE))  # [B,512]
        loss = symmetric_infonce_loss(e_A, e_V, tau)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(audio_proj.parameters()) + list(video_proj.parameters()), 1.0
        )
        optimizer.step()
        total += loss.item()
    return total / len(loader)


# Three-phase training demo
crema_ds = SyntheticCREMADDataset(n_per_class=40)
crema_dl = DataLoader(crema_ds, batch_size=32, shuffle=True, drop_last=True)

a_proj = ProjectionHead(768,  512).to(DEVICE)   # AudioProjection
v_proj = ProjectionHead(2048, 512).to(DEVICE)   # VideoFrameProjection

phase_results = {'phase': [], 'epoch': [], 'loss': []}

# --- Phase 1: Cross-Modal InfoNCE ---
print('=== Phase 1: Cross-Modal InfoNCE (13 epochs, LR=1e-4) ===')
opt1 = torch.optim.AdamW(list(a_proj.parameters())+list(v_proj.parameters()),
                         lr=1e-4, weight_decay=0.01)
for ep in range(5):  # demo: 5 epochs
    l = stage1_infonce_epoch(crema_dl, a_proj, v_proj, opt1)
    phase_results['phase'].append(1); phase_results['epoch'].append(ep+1); phase_results['loss'].append(l)
    if (ep+1) % 2 == 0: print(f'  Phase1 Ep {ep+1} | InfoNCE: {l:.4f}')

# --- Phase 2: SupCon LR=1e-5 ---
print('\n=== Phase 2: SupCon refinement (10 epochs, LR=1e-5) ===')
opt2 = torch.optim.AdamW(a_proj.parameters(), lr=1e-5, weight_decay=0.01)
for ep in range(5):
    total2 = 0.0
    for audio_feat, _, labels in crema_dl:
        e = a_proj(audio_feat.to(DEVICE))
        loss = supcon_loss(e, labels.to(DEVICE))
        opt2.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(a_proj.parameters(), 1.0)
        opt2.step(); total2 += loss.item()
    l2 = total2 / len(crema_dl)
    phase_results['phase'].append(2); phase_results['epoch'].append(ep+1); phase_results['loss'].append(l2)
    if (ep+1) % 2 == 0: print(f'  Phase2 Ep {ep+1} | SupCon: {l2:.4f}')

# --- Phase 3: SupCon LR=5e-5 ---
print('\n=== Phase 3: SupCon LR=5e-5 (target ≥4/6 emotions) ===')
opt3 = torch.optim.AdamW(a_proj.parameters(), lr=5e-5, weight_decay=0.01)
for ep in range(5):
    total3 = 0.0
    for audio_feat, _, labels in crema_dl:
        e = a_proj(audio_feat.to(DEVICE))
        loss = supcon_loss(e, labels.to(DEVICE))
        opt3.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(a_proj.parameters(), 1.0)
        opt3.step(); total3 += loss.item()
    l3 = total3 / len(crema_dl)
    phase_results['phase'].append(3); phase_results['epoch'].append(ep+1); phase_results['loss'].append(l3)
    if (ep+1) % 2 == 0: print(f'  Phase3 Ep {ep+1} | SupCon: {l3:.4f}')

# Plot three phases
fig, ax = plt.subplots(figsize=(9, 4))
colours = {1: 'steelblue', 2: 'orange', 3: 'green'}
labels_p = {1: 'Phase 1: InfoNCE (LR=1e-4)', 2: 'Phase 2: SupCon (LR=1e-5)', 3: 'Phase 3: SupCon (LR=5e-5)'}
for p in [1, 2, 3]:
    idx = [i for i,ph in enumerate(phase_results['phase']) if ph == p]
    ax.plot([phase_results['epoch'][i] for i in idx],
            [phase_results['loss'][i]  for i in idx],
            'o-', color=colours[p], label=labels_p[p], linewidth=2)
ax.set_title('Pipeline D — Three-Phase Training Arc (CREMA-D)')
ax.set_xlabel('Epoch within phase'); ax.set_ylabel('Loss')
ax.legend(fontsize=9); ax.grid(True, alpha=.3)
plt.tight_layout(); plt.show()

## Algorithms 20 & 21 — Pipeline E: CMU-MOSI Multimodal Sentiment Fusion

### Technical description

**Multi-Task Sentiment Loss:**
$$\mathcal{L}(\hat{y}, y, b) = \alpha \cdot \mathcal{L}_{\text{MSE}}(\hat{y}, y) + \beta \cdot \mathcal{L}_{\text{BCE}}(\hat{y}, b), \quad \alpha=0.6,\;\beta=0.4$$

**Numerical Stability of Logit BCE:**
$$\mathcal{L}_{\text{BCE}}(\hat{y}, b) = \max(0,\hat{y}) - b\hat{y} + \log(1 + e^{-|\hat{y}|})$$

**Evaluation Metrics:**
$$\text{MAE} = \frac{1}{N}\sum|\hat{y}_i - y_i|, \qquad r = \frac{\sum(\hat{y}_i-\bar{\hat{y}})(y_i-\bar{y})}{\sqrt{\sum(\hat{y}_i-\bar{\hat{y}})^2 \cdot \sum(y_i-\bar{y})^2}}$$
$$\text{Acc}_2 = \frac{1}{N}\sum\mathbf{1}[\text{sign}(\hat{y}_i) = \text{sign}(y_i)]$$

**Warmup-Cosine LR:**
$$\eta(t) = \begin{cases} \eta_0 \cdot t/T_w & t < T_w \\ \frac{\eta_0}{2}(1 + \cos\frac{\pi(t-T_w)}{T-T_w}) & t \geq T_w \end{cases}$$

### Plain language
Pipeline E gives the AI a sentiment score for a person's video opinion. It reads the transcript (text), listens to the voice (audio), and watches the face (video) simultaneously. The Transformer acts as a meeting room where each specialist updates their view after reading the others' reports. The dual loss trains it to be accurate on a sliding scale (MAE) AND to correctly classify positive vs. negative (binary).

In [ ]:
# Algorithms 20 & 21: Pipeline E — CMU-MOSI Multimodal Sentiment Fusion
# ----------------------------------------------------------------------

class SyntheticMOSIDataset(Dataset):
    """Proxy for CMU-MOSI: (e_text, e_audio, e_video, y ∈[-3,3], b ∈{0,1})."""
    def __init__(self, n=400):
        self.e_T = F.normalize(torch.randn(n, 512), dim=-1)
        self.e_A = F.normalize(torch.randn(n, 512), dim=-1)
        self.e_V = F.normalize(torch.randn(n, 512), dim=-1)
        self.y   = torch.rand(n) * 6 - 3       # ∈ [-3, 3]
        self.b   = (self.y > 0).float()         # binary label
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.e_T[i], self.e_A[i], self.e_V[i], self.y[i], self.b[i]


def multitask_loss(y_hat: torch.Tensor, y: torch.Tensor, b: torch.Tensor,
                   alpha: float = 0.6, beta: float = 0.4) -> torch.Tensor:
    """Multi-task loss: MSE (regression) + BCE (binary classification)."""
    L_mse = F.mse_loss(y_hat, y)
    L_bce = F.binary_cross_entropy_with_logits(y_hat, b)  # numerically stable
    return alpha * L_mse + beta * L_bce


@torch.no_grad()
def evaluate_pipeline_e(loader, model) -> Dict[str, float]:
    model.eval()
    preds, trues, bins = [], [], []
    for e_T, e_A, e_V, y, b in loader:
        y_hat = model(e_T.to(DEVICE), e_A.to(DEVICE), e_V.to(DEVICE))
        preds.append(y_hat.cpu()); trues.append(y); bins.append(b)
    P = torch.cat(preds); Y = torch.cat(trues); B = torch.cat(bins)
    mae   = (P - Y).abs().mean().item()
    p_bar = P - P.mean(); y_bar = Y - Y.mean()
    r = (p_bar * y_bar).sum() / (p_bar.norm() * y_bar.norm() + 1e-9)
    acc2  = ((P > 0) == (Y > 0)).float().mean().item()
    return {'MAE': mae, 'r': r.item(), 'Acc2': acc2}


def train_pipeline_e(n_epochs: int = 15) -> Tuple[MultimodalFusionTransformer, Dict]:
    """Algorithm 21: Full training loop with warmup-cosine LR."""
    ds_train = SyntheticMOSIDataset(300); ds_val = SyntheticMOSIDataset(100)
    dl_train = DataLoader(ds_train, batch_size=32, shuffle=True, drop_last=True)
    dl_val   = DataLoader(ds_val,   batch_size=32)

    model = MultimodalFusionTransformer().to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    T_total = len(dl_train) * n_epochs
    T_warm  = T_total // 10
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=T_total-T_warm, eta_min=1e-6)

    history = {'train_loss': [], 'val_MAE': [], 'val_Acc2': []}
    best_mae, best_state = float('inf'), None

    step = 0
    for ep in range(n_epochs):
        model.train(); ep_loss = 0.0
        for e_T, e_A, e_V, y, b in dl_train:
            e_T=e_T.to(DEVICE); e_A=e_A.to(DEVICE); e_V=e_V.to(DEVICE)
            y=y.to(DEVICE); b=b.to(DEVICE)

            y_hat = model(e_T, e_A, e_V)                  # Algorithm 20 forward
            loss  = multitask_loss(y_hat, y, b)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            # Warmup: linear for first T_warm steps, then cosine
            if step < T_warm:
                for g in opt.param_groups:
                    g['lr'] = 1e-4 * (step + 1) / T_warm
            else:
                scheduler.step()
            step += 1; ep_loss += loss.item()

        avg_loss = ep_loss / len(dl_train)
        metrics  = evaluate_pipeline_e(dl_val, model)
        history['train_loss'].append(avg_loss)
        history['val_MAE'].append(metrics['MAE'])
        history['val_Acc2'].append(metrics['Acc2'])

        if metrics['MAE'] < best_mae:
            best_mae = metrics['MAE']
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (ep+1) % 5 == 0:
            print(f'  Ep {ep+1:2d}/{n_epochs} | loss={avg_loss:.4f} | '
                  f'val_MAE={metrics["MAE"]:.4f} | val_Acc2={metrics["Acc2"]*100:.1f}%')

    if best_state:
        model.load_state_dict(best_state)
    print(f'\n  Best val_MAE: {best_mae:.4f} (target < 1.0)')
    return model, history


print('=== Algorithms 20-21: Pipeline E — CMU-MOSI Sentiment Fusion ===')
fusion_trained, history_e = train_pipeline_e(n_epochs=15)

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history_e['train_loss'], 'b-o', label='Train loss', linewidth=2)
ax1.set_title('Pipeline E — Training Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True, alpha=.3)
ax2.plot(history_e['val_MAE'],  'r-o', label='Val MAE', linewidth=2)
ax2.axhline(1.0, color='orange', linestyle='--', label='MAE target < 1.0')
ax2.set_title('Pipeline E — Validation MAE'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True, alpha=.3)
plt.tight_layout(); plt.show()

## Algorithm 22 — Domain Hot-Swap

### Technical description

**Theorem (Domain Switch Efficiency):** Under the Domain-Agnostic Shared Space Axiom, switching domain requires reloading only the FAISS index ($\approx$1-3s) and KG pickle ($\approx$0.5-2s), reducing domain-swap latency by $10$-$20\times$ compared to full model reload (50-90s for BERT + Wav2Vec).

**Full reload time:** $\frac{50+30}{3+1} \approx 20\times$ speedup (ratio of full-reload to index-only reload times).

**What stays frozen (never reloaded):**
- BERT + TextProjection $\checkmark$
- Wav2Vec 2.0 + AudioProjection $\checkmark$
- ResNet-50 backbone $\checkmark$

**What changes per domain:**
- FAISS index (COCO vs. ROCO vectors)
- Knowledge graph (general PMI vs. RadLex PMI)
- NER model (en_core_web_sm vs. en_core_sci_sm)
- Confidence threshold ($\tau=0.40$ vs. $\tau=0.55$)

### Plain language
Hot-swap is like swapping a library catalogue without re-reading any books. The 'reading brain' (BERT, Wav2Vec, ResNet) is identical for both domains — only the catalogue (FAISS + KG) and the librarian's rulebook (NER model + threshold) change. This takes 3-4 seconds instead of 60-90 seconds.

In [ ]:
# Algorithm 22: Domain Hot-Swap
# -----------------------------
# Input:  new_domain D', current pipeline
# Output: pipeline with updated FAISS + KG + NER (shared encoders UNCHANGED)

from dataclasses import dataclass
from typing import Optional

@dataclass
class DomainConfig:
    name:       str
    tau:        float       # confidence threshold
    top_K:      int = 20   # FAISS shortlist
    top_N:      int = 5    # final context
    ner_model:  str = 'en_core_web_sm'
    # In production: faiss_index_path, kg_path, corpus_metadata_path

GENERAL_DOMAIN = DomainConfig('general', tau=0.40, ner_model='en_core_web_sm')
MEDICAL_DOMAIN = DomainConfig('medical', tau=0.55, ner_model='en_core_sci_sm')


class MIVAKnightPipeline:
    """Full pipeline with hot-swap capability."""

    def __init__(self, proj_text: ProjectionHead, kg_general: Dict, kg_medical: Dict):
        self.proj_text   = proj_text
        self.current_domain = GENERAL_DOMAIN
        self.kg_store = {'general': kg_general, 'medical': kg_medical}
        self.active_kg   = kg_general

        # Synthetic FAISS-like corpus for two domains
        self.corpus_store = {
            'general': [{'text': 'a dog runs in the park'},
                        {'text': 'two cats sit on a mat'},
                        {'text': 'children play football'}],
            'medical': [{'text': 'pleural effusion on chest xray'},
                        {'text': 'cardiomegaly and pulmonary edema'},
                        {'text': 'pneumothorax consolidation findings'}]
        }
        self.active_corpus = self.corpus_store['general']

    def swap_domain(self, new_domain: DomainConfig) -> float:
        """Algorithm 22: hot-swap FAISS index + KG + NER only."""
        t0 = time.time()
        self.current_domain = new_domain
        # Reload KG (pickle)
        self.active_kg     = self.kg_store.get(new_domain.name, {})
        # Reload corpus / FAISS index
        self.active_corpus = self.corpus_store.get(new_domain.name, [])
        # NER model switch (simulated)
        elapsed = time.time() - t0
        print(f'  [Hot-swap] {new_domain.name} domain loaded in {elapsed*1000:.1f} ms '
              f'(τ={new_domain.tau}, NER={new_domain.ner_model})')
        print(f'  [Frozen]  TextProjection, BERT, AudioProjection, Wav2Vec — UNCHANGED')
        return elapsed

    def query(self, text: str) -> Dict:
        """End-to-end query with auto domain routing."""
        # Phase 1: Encode
        e_q, E_q, hint = text_encoder_v2(text, self.proj_text)
        # Auto domain routing
        if hint == 'medical' and self.current_domain.name != 'medical':
            self.swap_domain(MEDICAL_DOMAIN)
        elif hint is None and self.current_domain.name == 'medical':
            self.swap_domain(GENERAL_DOMAIN)
        # Phase 2: Retrieval (proxy)
        shortlist = [
            {'text': d['text'], 'dense_score': random.uniform(0.35, 0.85)}
            for d in self.active_corpus
        ]
        top_docs = ner_kg_rerank(shortlist, E_q, self.active_kg, top_n=2)
        s_max = max((d['final_score'] for d in top_docs), default=0.0)
        tier  = confidence_tier(s_max, self.current_domain.name)
        return {'text': text, 'domain': self.current_domain.name, 'tier': tier, 's_max': s_max}


# Demo hot-swap
kg_medical_demo = {'effusion': {'consolidation': 2.1, 'cardiomegaly': 1.8},
                   'pneumonia': {'consolidation': 3.0, 'pleural': 1.5}}
pipeline = MIVAKnightPipeline(ProjectionHead(768, 512), kg, kg_medical_demo)

print('=== Algorithm 22: Domain Hot-Swap ===')
test_qs = [
    "what dog breeds are good for families",
    "pleural effusion and pneumonia consolidation chest xray",
    "how do I train a dog to sit"
]
for q in test_qs:
    res = pipeline.query(q)
    print(f'  Query: "{q[:55]}"')
    print(f'         domain={res["domain"]} | tier={res["tier"]} | s_max={res["s_max"]:.3f}\n')

## Complete System: End-to-End MIVA-KNIGHT Demo

### Technical description

**Full 4-phase inference pipeline:**
$$\underbrace{\text{Input}}_{\text{text or audio}} \xrightarrow{\text{Phase 1}} \underbrace{e_q \in \mathbb{S}^{511}}_{\text{512-d embedding}} \xrightarrow{\text{Phase 2}} \underbrace{\mathcal{D}^* = \{d_1,\ldots,d_N\}}_{\text{top-N context}} \xrightarrow{\text{Phase 3}} \underbrace{\text{Answer}}_{\text{grounded text}} \xrightarrow{\text{Phase 4}} \underbrace{\text{Audio}}_{\text{gTTS TTS}}$$

**Precision@K evaluation:**
$$P@K = \frac{1}{|Q|}\sum_{q \in Q}\mathbf{1}[\exists r \in \mathcal{R}_q : r \in \text{top-}K(q)]$$

This final cell brings all 22 algorithms together in a single evaluation sweep.

### Plain language
The complete system combines all modules: the text encoder from Algorithm 7, the KG from Algorithm 9, the re-ranker from Algorithm 12, the confidence tier from Algorithm 13, and the hot-swap from Algorithm 22. We evaluate everything with Precision@K — the fraction of queries where the correct document appears in the top-K results.

In [ ]:
# Complete System Evaluation: P@K across all algorithms
# -------------------------------------------------------

def evaluate_precision_at_k(
    query_pairs: List[Tuple[str, List[str]]],  # (query_text, list_of_relevant_doc_ids)
    corpus: List[Dict],                         # [{id, text}]
    kg: Dict,
    proj: ProjectionHead,
    k_vals: List[int] = [1, 3, 5]
) -> Dict[str, float]:
    """Compute P@K over a query set."""
    hits = {k: 0 for k in k_vals}

    for query_text, relevant_ids in query_pairs:
        # Encode query
        e_q, E_q, _ = text_encoder_v2(query_text, proj)

        # Simulate FAISS + KG retrieval
        shortlist = [{'text': d['text'], 'id': d['id'],
                      'dense_score': float((e_q * F.normalize(torch.randn(512), dim=0)).sum()) + 0.5}
                     for d in corpus]
        top_all = ner_kg_rerank(shortlist, E_q, kg, top_n=max(k_vals))

        # Check hits
        retrieved_ids = [d.get('id', '') for d in top_all]
        for k in k_vals:
            if any(rid in relevant_ids for rid in retrieved_ids[:k]):
                hits[k] += 1

    n = max(len(query_pairs), 1)
    return {f'P@{k}': hits[k] / n for k in k_vals}


# Build evaluation dataset
eval_corpus = [
    {'id': f'doc_{i}', 'text': cap}
    for i, cap in enumerate([
        "a dog runs through the park with a leash",
        "cats and dogs playing outside",
        "the park has trees benches and dogs",
        "a bicycle leans against the park fence",
        "children play football in the sunny park",
        "two dogs fetch a ball in the garden",
        "a golden retriever with its owner",
        "traffic on a busy city street",
    ])
]

eval_queries = [
    ("dog running in park",     ['doc_0', 'doc_2', 'doc_5', 'doc_6']),
    ("bicycle transportation",  ['doc_3']),
    ("children sports outdoor", ['doc_4']),
    ("cat dog animals",         ['doc_1']),
]

eval_proj = ProjectionHead(768, 512)
pak_results = evaluate_precision_at_k(eval_queries, eval_corpus, kg, eval_proj)

print('=== Complete System Evaluation: Precision@K ===')
for metric, val in pak_results.items():
    bar = '█' * int(val * 40)
    print(f'  {metric}: {bar:<40} {val*100:.1f}%')

# Final summary visualisation
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Similarity matrix heatmap
B_viz = 6
e1_viz = F.normalize(torch.randn(B_viz, 512), dim=-1)
e2_viz = F.normalize(torch.randn(B_viz, 512), dim=-1)
sim_mat = (e1_viz @ e2_viz.T).detach().numpy()
ax = axes[0]
im = ax.imshow(sim_mat, cmap='Blues', vmin=-1, vmax=1)
ax.set_title('InfoNCE Sim Matrix $S_{ij}$', fontsize=11)
ax.set_xlabel('Modality B index'); ax.set_ylabel('Modality A index')
plt.colorbar(im, ax=ax, fraction=0.046)

# 2. Centroid visualisation (PCA 2D)
ax = axes[1]
centroids_np = centroids.detach().numpy()  # [8, 512]
U2, S2, _ = np.linalg.svd(centroids_np - centroids_np.mean(0), full_matrices=False)
c2d = U2[:, :2] * S2[:2]
colours_emo = plt.cm.Set1(np.linspace(0, 0.9, 8))
for i, (emo, col) in enumerate(zip(RAVDESS_EMOTIONS, colours_emo)):
    ax.scatter(*c2d[i], color=col, s=120, zorder=3, edgecolors='k')
    ax.annotate(emo, c2d[i], textcoords='offset points', xytext=(5,3), fontsize=7)
ax.set_title('Emotion Centroids (PCA 2D)', fontsize=11)
ax.grid(True, alpha=.3)

# 3. P@K bar chart
ax = axes[2]
k_labels = list(pak_results.keys())
k_values = list(pak_results.values())
bars = ax.bar(k_labels, [v*100 for v in k_values], color=['#1f77b4','#ff7f0e','#2ca02c'], edgecolor='k')
for bar, val in zip(bars, k_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val*100:.0f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Retrieval Quality: P@K', fontsize=11)
ax.set_ylabel('Precision (%)'); ax.set_ylim(0, 115)
ax.grid(True, alpha=.3, axis='y')

plt.suptitle('MIVA-KNIGHT Complete System — All 22 Algorithms', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('\n✓ All 22 algorithms implemented and demonstrated successfully.')

## Summary Table of All Algorithms

| # | Algorithm | Input | Output | Key equation |
|---|---|---|---|---|
| 1 | Joint Multimodal Repr. | $\{x_k\}$ | $\mathbf{r} = \phi(\mathbf{e}_1,\ldots,\mathbf{e}_K)$ | $\mathbf{r} = [\mathbf{e}_1;\ldots;\mathbf{e}_K]$ |
| 2 | Coordinated Repr. (InfoNCE) | Paired $\{(x_A,x_B)\}$ | $(P_A^*, P_B^*)$ on $\mathbb{S}^{511}$ | $\mathcal{L}_{\text{InfoNCE}} = \frac{1}{2}(\mathcal{L}_{A\to B}+\mathcal{L}_{B\to A})$ |
| 3 | Missing Modality | $\{x_k\}$ (some None) | Robust $\mathbf{r}$ | $\mathbf{e}_k = \mathbf{0}$ if absent |
| 4 | Cross-Modal Transformer | $\{\hat{e}_k\}$ | $\hat{y}$ | Pre-LN $x' = x + \text{Attn}(\text{LN}(x))$ |
| 5 | Early Fusion | $\{x_k\}$ | $\hat{y} = g([\mathbf{e}_1;\ldots])$ | Pre-model concat |
| 6 | Late Fusion | $\{x_k\}$ (some None) | $\hat{y} = \sum w_k g_k(x_k)$ | Re-normalise missing |
| 7 | TextEncoderV2 | text $t$ | $\hat{e}_q \in \mathbb{S}^{511}$, $\mathcal{E}_q$ | BERT mean-pool + projection |
| 8 | Pipeline A Epoch | COCO loader | Updated $(P_T, P_I)$ | InfoNCE + AdamW + cosine LR |
| 9 | PMI KG Build | Captions | KG dict (PMI weights) | $\text{PMI}(a,b)=\log\frac{c_{ab}N}{f_a f_b}$ |
| 10 | KG DFS Search | entity, KG | Neighbour triples | 2-hop depth-first |
| 11 | Surgical Transfer (ROCO) | ROCO pairs | $P_I^{\phi^*}$ (2.23% params) | $\mathcal{L}_{\text{InfoNCE}} + \lambda\mathcal{L}_{\text{distill}}$ |
| 12 | KG Re-ranker | Shortlist, $\mathcal{E}_q$ | Top-N docs, final score | $\alpha\tilde{s}_{\text{dense}} + (1-\alpha)\tilde{s}_{\text{KG}}$ |
| 13 | Full Query Pipeline | user input | answer + tier | tier(s)∈{full,partial,rejected} |
| 14 | RAVDESS Cache | wav files | 768-d embeddings | Wav2Vec mean-pool |
| 15 | SupCon Loss | $\hat{E}$, labels | scalar loss | $\mathcal{L}_{\text{SupCon}}$ |
| 16 | SupCon Epoch | RAVDESS loader | Updated $f_\theta$ | SupCon + AdamW |
| 17 | Centroid-NN Eval | loader, $f_\theta$ | accuracy, centroids | $\hat{y} = \arg\max_c \hat{e}\cdot\bar{c}_c$ |
| 18 | Middle-Frame Extract | mp4, frame count | pool5 features $[2048]$ | $f^* = \lfloor N/2 \rfloor$ |
| 19 | Pipeline D Stage 1 | CREMA-D loader | $(f_\theta^A, f_\phi^V)$ aligned | Cross-modal InfoNCE |
| 20 | Fusion Transformer FWD | $(\hat{e}_T,\hat{e}_A,\hat{e}_V)$ | $\hat{y} \in \mathbb{R}$ | Pre-LN Transformer |
| 21 | Pipeline E Training | MOSI loader | Trained model, MAE=0.7238 | MSE+BCE multi-task |
| 22 | Domain Hot-Swap | new domain $D'$ | Updated FAISS+KG+NER | SharedEncoders UNCHANGED |